# Silver

In [0]:
CATALOG = "crop_risk"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType
import pyspark.sql.functions as F

In [0]:
bronze_crop_trend = spark.table(f"{CATALOG}.bronze.crop_trend_master")
bronze_crop_trend.show(10)

In [0]:
bronze_crop_trend.select("risk_label").distinct().show()

In [0]:
key_cols = ["province", "crop", "year", "quarter"]
measure_col = "production"
 
bronze_crop_trend = spark.table(f"{CATALOG}.bronze.crop_trend_master")
other_cols = [c for c in bronze_crop_trend.columns if c not in key_cols + [measure_col]]
 
df_crop_trend = (
    bronze_crop_trend.groupBy(*key_cols)
    .agg(
        F.sum(measure_col).alias(measure_col),
        *[F.first(c, ignorenulls=True).alias(c) for c in other_cols],
    )
    .withColumn("_processed_at", F.current_timestamp())
)
 
(
    df_crop_trend.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.crop_trend_master")
)